## Background

## Imports

In [1]:
import os
import re
import warnings
import emoji
import numpy as np
import pandas as pd
import seaborn as sns
import spacy
import torch
from torch.utils.data import Dataset, random_split
from transformers import RobertaTokenizerFast, RobertaForTokenClassification, Trainer, TrainingArguments
from sklearn.metrics import f1_score
from tqdm.auto import tqdm
import pandas as pd

import matplotlib.pyplot as plt

/Users/k_akhynko/Desktop/work/thesis/manipulative-narrative-detection/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Constants

In [3]:
TRAIN_PATH = "../data/"
TRAIN_NAME = "train.parquet"

MODEL_NAME = "FacebookAI/xlm-roberta-large"

In [ ]:
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print(f"Using device: {DEVICE}")

Using device: mps


## Read data

In [5]:
df = pd.read_parquet(os.path.join(TRAIN_PATH, TRAIN_NAME))
df.head()

,id,content,lang,manipulative,techniques,trigger_words
0,0bb0c7fa-101b-4583-a5f9-9d503339141c,Новий огляд мапи DeepState від російського вій...,uk,True,"[euphoria, loaded_language]","[[27, 63], [65, 88], [90, 183], [186, 308]]"
1,7159f802-6f99-4e9d-97bd-6f565a4a0fae,Недавно 95 квартал жёстко поглумился над русск...,ru,True,"[loaded_language, cherry_picking]","[[0, 40], [123, 137], [180, 251], [253, 274]]"
2,e6a427f1-211f-405f-bd8b-70798458d656,🤩\nТим часом йде евакуація Бєлгородського авто...,uk,True,"[loaded_language, euphoria]","[[55, 100]]"
3,1647a352-4cd3-40f6-bfa1-d87d42e34eea,В Україні найближчим часом мають намір посилит...,uk,False,None,None
4,9c01de00-841f-4b50-9407-104e9ffb03bf,"Расчёты 122-мм САУ 2С1 ""Гвоздика"" 132-й бригад...",ru,True,[loaded_language],"[[114, 144]]"


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3822 entries, 0 to 3821
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   id             3822 non-null   object
 1   content        3822 non-null   object
 2   lang           3822 non-null   object
 3   manipulative   3822 non-null   bool  
 4   techniques     2589 non-null   object
 5   trigger_words  2589 non-null   object
dtypes: bool(1), object(5)
memory usage: 153.2+ KB


In [7]:
# If there are no manipulations in the text, we will set techniques and trigger_words to empty lists
df['techniques'] = df['techniques'].apply(lambda x: [] if x is None else x)
df['trigger_words'] = df['trigger_words'].apply(lambda x: [] if x is None else x)

## Dataset

In [8]:
tokenizer = RobertaTokenizerFast.from_pretrained(MODEL_NAME)

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'XLMRobertaTokenizerFast'. 
The class this function is called from is 'RobertaTokenizerFast'.


In [9]:
class SpansDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=512):
        self.tokenizer = tokenizer
        self.texts = df['content']
        self.spans = df['trigger_words']
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        spans = self.spans[idx]
        encoding = self.tokenizer(text, padding='max_length', truncation=True, max_length=self.max_length, return_offsets_mapping=True)
        input_ids = encoding['input_ids']
        attention_mask = encoding['attention_mask']
        offset_mapping = encoding['offset_mapping']

        # Initialize token-level labels.
        # We'll mark tokens that are not real (i.e. padded tokens) as -100.
        token_labels = torch.full((self.max_length,), -100, dtype=torch.long)
        # For tokens that are not padding, set default label 0 (non-manipulative).
        for i in range(self.max_length):
            if attention_mask[i] == 1:
                token_labels[i] = 0

        # Loop over each token using its offset mapping.
        # If the token (defined by its character span) overlaps with any trigger span, label it as 1.
        # print(self.data.iloc[idx]['trigger_words_phrases'])
        for i, (token_start, token_end) in enumerate(offset_mapping):
            # Skip pad tokens
            if attention_mask[i] == 0:
                continue
            for span in spans:
                span_start, span_end = span
                # Check if there is any overlap between token span and trigger span.
                if token_end > span_start and token_start < span_end:
                    # print(text[token_start:token_end])
                    token_labels[i] = 1
                    break  # No need to check other spans for this token.

        return {
            'input_ids': torch.tensor(input_ids, dtype=torch.long),
            'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
            'token_labels': torch.tensor(token_labels, dtype=torch.long)
            }

In [10]:
dataset = SpansDataset(df, tokenizer)

In [11]:
# Split dataset into training and validation sets
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

## Modeling

In [12]:
# Load model
model = RobertaForTokenClassification.from_pretrained(MODEL_NAME, num_labels=2)

You are using a model of type xlm-roberta to instantiate a model of type roberta. This is not supported for all configurations of models and can yield errors.
Some weights of RobertaForTokenClassification were not initialized from the model checkpoint at FacebookAI/xlm-roberta-large and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [13]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = torch.sigmoid(torch.tensor(logits)).cpu().numpy()
    return {"f1": f1_score(labels.flatten(), predictions.flatten(), average="macro")}

In [14]:
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

/Users/k_akhynko/Desktop/work/thesis/manipulative-narrative-detection/venv/lib/python3.12/site-packages/transformers/training_args.py:1594: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/Users/k_akhynko/Desktop/work/thesis/manipulative-narrative-detection/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/var/folders/tv/s_85wvjj5h15ffbhxgbrszlw0000gn/T/ipykernel_5134/3655612082.py:45: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  'token_labels': torch.tensor(token_labels, dtype=torch.long)

KeyboardInterrupt



In [ ]:
trainer.evaluate()

## Inference

In [18]:
from tqdm import tqdm

def predict_manipulation_spans(text, model, tokenizer, max_length=512, token_threshold=0.5, device=DEVICE):
    predictions = []

    model.to(device)
    model.eval()

    encoding = tokenizer(
        text,
        padding='max_length',
        truncation=True,
        max_length=max_length,
        return_tensors="pt",
        return_offsets_mapping=True
    )
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)
    offset_mapping = encoding['offset_mapping'][0].tolist()

    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)

    token_logits = outputs["token_logits"].squeeze(-1)  # Now shape: (batch_size, seq_length)
    token_probs = torch.sigmoid(torch.tensor(token_logits)).cpu().numpy()
    token_preds = (token_probs > token_threshold)
    # Mask out padded tokens
    token_preds = token_probs * attention_mask.cpu().numpy()

    # For span reconstruction, assume batch size 1.
    token_preds = token_preds[0].tolist()  # List of predictions per token.

    # Convert token predictions into spans using the offset mapping.
    spans = []
    current_span = None
    for pred, (tok_start, tok_end) in zip(token_preds, offset_mapping):
        if pred == 1:
            if current_span is None:
                current_span = [tok_start, tok_end]
            else:
                # Extend the span to the end of this token.
                current_span[1] = tok_end
        else:
            if current_span is not None:
                spans.append(tuple(current_span))
                current_span = None
    # Append any remaining span.
    if current_span is not None:
        spans.append(tuple(current_span))

    return {
        "token_probs": token_probs,
        "token_preds": token_preds,
        "spans": spans
    }

In [ ]:
# Predicting for test

predictions = []
for text in tqdm(test_dataset.content.values):
    predictions.append(run_inference(model, tokenizer, text))

In [ ]:
# Saving prediction for token:
pred_token = pd.DataFrame({
    "id": test.id.values
})

pred_values = [p["spans"] for p in predictions]
pred_token["trigger_words"] = pred_values

In [19]:
# Compute metrics
#  ....................................